Practical No: 2

For a given set of training data examples stored in a .CSV file, implement and demonstrate the Candidate-Elimination algorithm to output a description of the set of all hypotheses consistent with the training examples

In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv('enjoysport1.csv')

In [3]:
concepts = np.array(data.iloc[:,0:-1])
print("\nInstances are:\n",concepts)


Instances are:
 [['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
 ['rainy' 'warm' 'high' 'strong' 'warm' 'same']
 ['rainy' 'cold' 'high' 'strong' 'warm' 'change']
 ['sunny' 'warm' 'high' 'strong' 'cool' 'change']]


In [4]:
target = np.array(data.iloc[:,-1])
print("\nTarget Values are: ",target)


Target Values are:  ['yes' 'yes' 'no' 'yes']


In [5]:
specific_hypothesis = ["Φ"] * concepts.shape[1]
general_hypothesis = [["?"] * concepts.shape[1]]

print("Initial Specific Hypothesis (S):", specific_hypothesis)
print("Initial General Hypothesis (G):", general_hypothesis)
print("-" * 60)

Initial Specific Hypothesis (S): ['Φ', 'Φ', 'Φ', 'Φ', 'Φ', 'Φ']
Initial General Hypothesis (G): [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------


In [6]:
def is_consistent(h, x):
    """Checks if hypothesis h is consistent with instance x."""
    for i in range(len(h)):
        if h[i] == '?':
            continue
        if h[i] != x[i]:
            return False
    return True

In [7]:
def is_more_general_than_or_equal(h1, h2):
    """Checks if h1 is more general than or equal to h2."""
    for i in range(len(h1)):
        if h1[i] == '?':
            continue
        elif h2[i] == '?': 
            return False
        elif h1[i] != h2[i]:
            return False
    return True


In [8]:
def remove_redundant(hypotheses, type='S'):
    """Removes redundant hypotheses from a boundary set (S or G)."""
    filtered_hypotheses = []
    for h1 in hypotheses:
        is_redundant = False
        for h2 in hypotheses:
            if h1 != h2:
                if type == 'S' and is_more_general_than_or_equal(h2, h1): # In S, remove h1 if h2 is more general
                    is_redundant = True
                    break
                elif type == 'G' and is_more_general_than_or_equal(h1, h2): # In G, remove h1 if h2 is more general
                    is_redundant = True
                    break
        if not is_redundant:
            filtered_hypotheses.append(h1)
    return filtered_hypotheses

In [9]:
def run_candidate_elimination(concepts, target):
    num_attributes = concepts.shape[1]

    S = []
    G = [['?'] * num_attributes]

    for i, instance in enumerate(concepts):
        label = target[i].lower()
        print(f"\n--- Processing example {i+1}: Instance={instance}, Label={label} ---")

        if label == 'yes':
            G = [g for g in G if is_consistent(g, instance)]

            S_to_process = list(S)
            new_S_hypotheses = []

            if not S:
                new_S_hypotheses.append(list(instance))
            else:
                for s in S_to_process:
                    if not is_consistent(s, instance):
                        temp_s = list(s)
                        for attr_idx in range(num_attributes):
                            if temp_s[attr_idx] == 'Φ':
                                temp_s[attr_idx] = instance[attr_idx]
                            elif temp_s[attr_idx] != instance[attr_idx]:
                                temp_s[attr_idx] = '?'
                        
                        if is_consistent(temp_s, instance) and any(is_more_general_than_or_equal(g_hyp, temp_s) for g_hyp in G):
                            new_S_hypotheses.append(temp_s)
                    else:
                        new_S_hypotheses.append(s)
            S = new_S_hypotheses
            S = remove_redundant([list(h) for h in set(tuple(h_tuple) for h_tuple in S)], type='S')

        elif label == 'no':
            S = [s for s in S if not is_consistent(s, instance)]

            G_to_process = list(G)
            new_G_hypotheses = []

            for g in G_to_process:
                if is_consistent(g, instance):
                    for attr_idx in range(num_attributes):
                        if g[attr_idx] == '?':
                            for s_hyp in S:
                                if s_hyp[attr_idx] != '?' and s_hyp[attr_idx] != instance[attr_idx]:
                                    specialized_g = list(g)
                                    specialized_g[attr_idx] = s_hyp[attr_idx]
                                    
                                    if any(is_more_general_than_or_equal(specialized_g, s_cand) for s_cand in S) and \
                                       not is_consistent(specialized_g, instance):
                                        new_G_hypotheses.append(specialized_g)
                else:
                    new_G_hypotheses.append(g)
            G = new_G_hypotheses
            G = remove_redundant([list(h) for h in set(tuple(h_tuple) for h_tuple in G)], type='G')

        S = [list(h) for h in sorted(list(set(tuple(h_tuple) for h_tuple in S)))]
        G = [list(h) for h in sorted(list(set(tuple(h_tuple) for h_tuple in G)))]

        print("S boundary:", S)
        print("G boundary:", G)

    return S, G

final_S, final_G = run_candidate_elimination(concepts, target)

print("\n========== FINAL VERSION SPACE ==========")
print("Final Specific Hypothesis (S):")
for s_hyp in final_S:
    print(s_hyp)
print("\nFinal General Hypothesis (G):")
for g_hyp in final_G:
    print(g_hyp)
print("=========================================")


--- Processing example 1: Instance=['sunny' 'warm' 'normal' 'strong' 'warm' 'same'], Label=yes ---
S boundary: [['sunny', 'warm', 'normal', 'strong', 'warm', 'same']]
G boundary: [['?', '?', '?', '?', '?', '?']]

--- Processing example 2: Instance=['rainy' 'warm' 'high' 'strong' 'warm' 'same'], Label=yes ---
S boundary: [['?', 'warm', '?', 'strong', 'warm', 'same']]
G boundary: [['?', '?', '?', '?', '?', '?']]

--- Processing example 3: Instance=['rainy' 'cold' 'high' 'strong' 'warm' 'change'], Label=no ---
S boundary: [['?', 'warm', '?', 'strong', 'warm', 'same']]
G boundary: [['?', '?', '?', '?', '?', 'same'], ['?', 'warm', '?', '?', '?', '?']]

--- Processing example 4: Instance=['sunny' 'warm' 'high' 'strong' 'cool' 'change'], Label=yes ---
S boundary: [['?', 'warm', '?', 'strong', '?', '?']]
G boundary: [['?', 'warm', '?', '?', '?', '?']]

========== FINAL VERSION SPACE ==========
Final Specific Hypothesis (S):
['?', 'warm', '?', 'strong', '?', '?']

Final General Hypothesis (G):

In [10]:
print("FINAL RESULT")
print("Final Specific Hypothesis (S):", specific_hypothesis)
print("Final General Hypothesis (G):", general_hypothesis)

FINAL RESULT
Final Specific Hypothesis (S): ['Φ', 'Φ', 'Φ', 'Φ', 'Φ', 'Φ']
Final General Hypothesis (G): [['?', '?', '?', '?', '?', '?']]
